# Scube Mouse Hypothalamus (2/2): Alignment of consecutive ST slices

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

In [2]:
import SPACEL
from SPACEL import Scube

In [3]:
seed = 2
from SPACEL.setting import set_environ_seed
set_environ_seed(seed)

Setting environment seed: 2


## Spatial transcriptomic data import

The input data type of Scube is a list of anndata objects for each slice. The anndata objects of each slice must contain a column of spatial domain for alignment in ``.obs``.

In [ ]:
adata = sc.read(f'./{seed}/DLPFC_151507_151510_Splane.h5ad')

adata[0].obs

,in_tissue,array_row,array_col,original_clusters,0,1,2,3,4,5,6,slice_id,spatial_domain
AAACAACGAATAGTTC-1,1,0,16,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,151507,2


In [5]:
slices = ['151507', '151508', '151509', '151510']

adata_list=[]

for i in range(4):
    adata_list.append(adata[adata.obs['slice_id'] == slices[i]])
    adata_list[i].obs.spatial_domain = adata_list[i].obs.spatial_domain.astype('category')

In [6]:
adata_list[1]

AnnData object with n_obs × n_vars = 4381 × 33538
    obs: 'in_tissue', 'array_row', 'array_col', 'original_clusters', '0', '1', '2', '3', '4', '5', '6', 'slice_id', 'spatial_domain'
    obsm: 'spatial'

Plot all raw slices stacked in a single figure. Cells colored by spatial domain.First, we define the color of each spatial domain.

## Pairwise alignment

Here, we perform pairwise alignment of slices. The ``cluster_key`` is a column of ``.obs`` which contain spatial domain information used for alignment. The ``n_neighbors`` affect how much neighbors are considered when calculate similarity of two slices. The ``knn_exclude_cutoff`` affect max distance of neighbors. By default, ``n_neighbors = 15`` and ``knn_exclude_cutoff = n_neighbors`` means a spot/cell in source slices consider 15 neighbors in target slices, and neighbor will be exclude when the distance of neighbors larger than the median of neareast 30 neighbors distance in all spot/cell in target slice. ``p`` affect the exponent of overlap penalty, and a larger p means stronger partial alignment capability.

In [ ]:
Scube.align(adata_list,
      cluster_key='spatial_domain', 
      n_neighbors = 15, 
      n_threads=10,
      p=2,
      seed=seed,
     )

Start alignment...
Alignment slice 1 to 0
Alignment slice 2 to 1
Alignment slice 3 to 2
Runtime: 183.96169900894165 s


After alignment, the new spots/cells location coordinates saved in ``.obsm['spatial_aligned']`` of anndata object of each slices. When ``write_loc_path`` provied by users, the location coordinates of all spots/cells of all slices will be saved in ``write_loc_path``.

## LAA & ARI

In [8]:
from sklearn.metrics import pairwise
from sklearn.metrics import adjusted_rand_score

# LAA
#calculate laywise bar, result needs to be devided by total number to calculate the accuracy rate
def cal_layer_based_alignment_result(alignment, s1, s2):
    #fei added
    labels = []
    labels.extend(s1.obs['original_clusters'])
    labels.extend(s2.obs['original_clusters'])

    res = []
    l_dict = {"0": 0, "1": 1, "2": 2, "3": 3, "4": 4, "5": 5, "6": 6}
    cnt0 = 0
    cnt1 = 0
    cnt2 = 0
    cnt3 = 0
    cnt4 = 0
    cnt5 = 0
    cnt6 = 0
    for i, elem in enumerate(alignment):
        if np.max(elem) - np.min(elem) != 0: # paste2 might have spots that are not aligned to any other spots. paste might have negative alignment
            if labels[i] == '-1' or labels[elem.argmax() + alignment.shape[0]] == '-1':
                continue
            if l_dict[labels[i]] == l_dict[labels[elem.argmax() + alignment.shape[0]]]:
                cnt0 += 1
            if abs(l_dict[labels[i]] - l_dict[labels[elem.argmax() + alignment.shape[0]]]) == 1:
                cnt1 += 1
            if abs(l_dict[labels[i]] - l_dict[labels[elem.argmax() + alignment.shape[0]]]) == 2:
                cnt2 += 1
            if abs(l_dict[labels[i]] - l_dict[labels[elem.argmax() + alignment.shape[0]]]) == 3:
                cnt3 += 1
            if abs(l_dict[labels[i]] - l_dict[labels[elem.argmax() + alignment.shape[0]]]) == 4:
                cnt4 += 1
            if abs(l_dict[labels[i]] - l_dict[labels[elem.argmax() + alignment.shape[0]]]) == 5:
                cnt5 += 1
            if abs(l_dict[labels[i]] - l_dict[labels[elem.argmax() + alignment.shape[0]]]) == 6:
                cnt6 += 1
    res.extend([cnt0, cnt1, cnt2, cnt3, cnt4, cnt5, cnt6])
    return res


def compute_alignment_ari(sliceA, sliceB, pi):
    mapped_clusters = []
    for j in range(pi.shape[1]):
        mapping = pi[:, j]
        #original: if np.sum(mapping) > 0:
        if np.max(mapping) - np.min(mapping) != 0:
            i = np.argmax(mapping)
            mapped_clusters.append(sliceA.obs['original_clusters'][i])
        else:
            mapped_clusters.append("NULL")
    assert len(sliceB.obs['original_clusters']) == len(mapped_clusters)
    true_clusters_mapped_region = []
    mapped_clusters_mapped_region = []
    for i in range(len(sliceB.obs['original_clusters'])):
        if mapped_clusters[i] != "NULL":
            true_clusters_mapped_region.append(sliceB.obs['original_clusters'][i])
            mapped_clusters_mapped_region.append(mapped_clusters[i])

    ari = adjusted_rand_score(true_clusters_mapped_region, mapped_clusters_mapped_region)
    return ari

In [ ]:
ari_values = []
laa_values = []

for i in range(1, len(slices)):
    target_ind = i-1
    source_ind = i
    f1 = adata_list[target_ind].obsm['spatial_aligned'].to_numpy()
    f2 = adata_list[source_ind].obsm['spatial_aligned'].to_numpy()

    alignment = pairwise.euclidean_distances(f1, f2)
    alignment = np.exp(-alignment)
    
    name1 = slices[target_ind].split('.')[-1]
    name2 = slices[source_ind].split('.')[-1]
    # Save as .pkl
    alignment_df = pd.DataFrame(alignment)
    alignment_df.to_pickle(f'./{seed}/{name1}_{name2}_AlignmentPi.pkl')
    
    res = cal_layer_based_alignment_result(alignment, adata_list[target_ind], adata_list[source_ind])
    divisor = alignment.shape[0]
    laa_result = [num / divisor for num in res]
    
    ari = compute_alignment_ari(adata_list[target_ind], adata_list[source_ind], alignment)
    ari_values.append(ari)
    laa_values.append(laa_result)
    print(laa_result, ari)

# Create DataFrame with desired format
header_row = [f"{slices[i-1]}_{slices[i]}" for i in range(1, len(slices))]
ari_df = pd.DataFrame([ari_values], columns=header_row)
laa_df = pd.DataFrame([laa_values], columns=header_row)

[0.830135039090263, 0.08813077469793888, 0.00023691068467187872, 0.0, 0.0, 0.0, 0.024638711205875383] 0.8047456958014124
[0.7130792056608081, 0.08377082857795023, 0.0, 0.0, 0.0, 0.0, 0.029901848892946815] 0.8065940534674352
[0.8414786967418546, 0.0941938178780284, 0.0, 0.0, 0.0, 0.0, 0.013993316624895571] 0.8178288213787376


In [ ]:
# Save as a CSV file
ari_df.to_csv(f'./{seed}/151507_151510_ARI.csv')
laa_df.to_csv(f'./{seed}/151507_151510_LAA.csv')

## Plot alignment results

Save all spots/cells aligned coordiantes and colors in a DataFrame.

In [11]:
c_dict = {'0': '#1f77b4',
 '1': '#ff7f0e',
 '2': '#2ca02c',
 '3': '#d62728',
 '4': '#9467bd',
 '5': '#8c564b',
 '6': '#e377c2'}

In [ ]:
coo = pd.DataFrame()

for i in range(len(adata_list)):
    loc = adata_list[i].obsm['spatial_aligned'].copy()
    loc['Z'] = i
    loc['slice'] = adata_list[i].obs['slice_id']
    loc['original_clusters'] = adata_list[i].obs['original_clusters']
    loc['celltype_colors'] = adata_list[i].obs['original_clusters'].replace(c_dict)
    coo = pd.concat([coo,loc],axis=0)
    
coo.to_csv(f'./{seed}/SPACEL_DLPFC_151507_151510_aligned_coordinates_colors.csv')

In [13]:
coo

,X,Y,Z,slice,original_clusters,celltype_colors
AAACAACGAATAGTTC-1,-3422.000000,-4081.000000,0,151507,1,#ff7f0e
AAACAAGTATCTCCCA-1,2480.000000,1925.000000,0,151507,3,#d62728
AAACAATCTACTAGCA-1,-1565.000000,-3717.000000,0,151507,1,#ff7f0e
AAACACCAATAACTGC-1,-3236.000000,2986.000000,0,151507,0,#1f77b4
AAACAGCTTTCAGAAG-1,-3919.000000,1068.000000,0,151507,6,#e377c2
...,...,...,...,...,...,...
TTGTTTCACATCCAGG-1,-1246.409668,322.683044,3,151510,4,#9467bd
TTGTTTCATTAGTCTA-1,-2071.458008,581.033020,3,151510,5,#8c564b
TTGTTTCCATACAACT-1,-2316.325195,-1220.782593,3,151510,3,#d62728
TTGTTTGTATTACACG-1,-1277.489014,2130.276611,3,151510,6,#e377c2


## Plot

In [ ]:
def plot_3d(
    loc,
    val=None,
    color=None,
    figsize=(12,12),
    return_fig=True,
    elev=None,
    azim=None,
    roll=None,
    xlim=None,
    ylim=None,
    zlim=None,
    frameon=True,
    save_path=None,
    save_dpi=150,
    show=True,
    *args,
    **kwargs
):
    """Plot all slices stacked in 3D

    Plot all slices stacked with a given number of subplots rows and columns. Spots/cells colored by spatial domain.

    Args:
        loc: An array of the coordinates of each spots/cells in all slices, which the first three columns are X-axis, Y-axis, Z-axis coordinates.
        val: The colors of each spots/cells given in loc use to plot.
        color: The colors of each spots/cells given in loc use to plot.
        figsize: Size of the figure.
        return_fig: Whether to return the figure.
        elev: The elevation angle in the vertical plane in degrees. If ``None`` then the initial value as specified in the ``Axes3D`` constructor is used.
        azim: The azimuth angle in the horizontal plane in degrees. If ``None`` then the initial value as specified in the ``Axes3D`` constructor is used.
        xlim: A tuple given the left and right xlims in X-axis.
        ylim: A tuple given the left and right xlims in Y-axis.
        zlim: A tuple given the left and right xlims in Z-axis.
        frameon: Whether to hide the coordinate axes.
        save_path: A string representing the path directory where the figure saved.
        save_dpi: The resolution in dots per inch.
        show: Whether to show the figure.

    Returns:
        A ``matplotlib.figure.Figure`` object
    """
    if 'marker' not in kwargs.keys():
        kwargs['marker'] = 'o'
    if 's' not in kwargs.keys():
        kwargs['s'] = 5
    if 'cmap' not in kwargs.keys():
        kwargs['cmap'] = 'Spectral_r'
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(projection='3d')
    if (elev is not None) and (azim is not None) and (roll is not None):
        ax.view_init(elev, azim, roll)
    if color is None:
        ax.scatter(loc[:,0], loc[:,1], loc[:,2],c=val,*args,**kwargs)
    else:
        ax.scatter(loc[:,0], loc[:,1], loc[:,2],c=color,*args,**kwargs)
    
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    if not frameon:
        plt.axis('off')
        plt.grid(b=None)
    if save_path is not None:
        print(save_path)
        plt.savefig(save_path,dpi=save_dpi,bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close()
    if return_fig:
        return fig

In [ ]:
seed = 1
df = pd.read_csv(f'./{seed}/SPACEL_DLPFC_151507_151510_aligned_coordinates_colors.csv')
coords = df[['X', 'Y', 'Z']]
# coords['Z'] = coords['Z']

In [ ]:
fig = plot_3d(coords.values, val=None, color=df['celltype_colors'], elev=30, azim=140, roll=0, frameon=False, show=True, s=35, save_dpi=300, return_fig=True) # Scube is from SPACEL

fig.savefig(f'./{seed}/SPACEL_gtAndScube_lymph_3dviz_angle30_140_s=35.png', bbox_inches="tight")